# Sustained Attention (Vigilance) — N-back Benchmark

Tests sustained attention via n-back monitoring with near-miss distractors over 140 items.

**Cognitive Science:** Kirchner (1958), Mackworth (1948)

## Cognitive Science Background

**Vigilance** (sustained attention) is the ability to maintain focus on a monitoring task over time (Warm, Parasuraman & Matthews, 2008). Performance typically degrades — the "vigilance decrement." This benchmark combines the **n-back paradigm** (Kirchner, 1958) with **vigilance decrement** design (Mackworth, 1948).

### Design Features

- **3-back and 4-back conditions** — tests working memory load under sustained attention
- **Near-miss distractors** — confusable letters (e.g., B/D/P) increase false alarm potential
- **Decreasing target rate** — targets become rarer over time, increasing vigilance demand
- **Signal detection scoring** — hit rate minus false alarm rate (d' proxy) as primary metric

Previous version scored 1.0 for all 10 models (ceiling effect). This redesign uses n-back with confusable distractors to break the ceiling.

## Scoring

$$\text{Score} = 0.35 \times \text{accuracy} + 0.35 \times \text{sensitivity} + 0.15 \times \text{vigilance\_resistance} + 0.15 \times (1 - \text{FA\_rate})$$

Weighted: 55% 3-back + 45% 4-back.

A score of 1.0 requires perfect hit rate, zero false alarms, and no vigilance decrement across both conditions.

**Expected ranges** (from Bedrock testing):
- Frontier models (Claude Opus 4.6): ~0.86
- Mid-tier (Nova Pro): ~0.59
- Small models (Ministral 3B): ~0.60

### References

- Kirchner (1958): N-back working memory paradigm
- Mackworth (1948): Clock test — vigilance decrement
- Parasuraman & Davies (1977): Vigilance taxonomy
- Warm, Parasuraman & Matthews (2008): Vigilance requires hard mental work

In [ ]:
!pip install -q protobuf==5.29.6 kaggle-benchmarks numpy 2>/dev/null

In [ ]:
"""
Vigilance benchmark stimuli — N-back sustained attention task.

Generates long sequences where the model must identify n-back matches
(current item same as item N positions back) among near-miss distractors.

Cognitive basis:
- Kirchner (1958): N-back task for working memory / sustained attention
- Mackworth (1948): Vigilance decrement over time
- Parasuraman & Davies (1977): Signal detection in sustained monitoring
"""

import random
import hashlib


# Stimulus pool: letters chosen to maximize confusability (near-miss distractors)
# Groups of visually/phonetically similar items increase false alarm potential
STIMULUS_POOL = list("BCDGPTVFHKLMNRSXZ")

# Confusable pairs — used to generate near-miss distractors
CONFUSABLE = {
    "B": ["D", "P"],
    "D": ["B", "G"],
    "G": ["C", "D"],
    "P": ["B", "T"],
    "T": ["P", "D"],
    "V": ["F", "B"],
    "F": ["V", "H"],
    "H": ["K", "F"],
    "K": ["H", "X"],
    "L": ["M", "N"],
    "M": ["N", "L"],
    "N": ["M", "L"],
    "R": ["S", "L"],
    "S": ["X", "Z"],
    "X": ["K", "S"],
    "Z": ["S", "X"],
    "C": ["G", "S"],
}


def generate_nback_sequence(
    seed: str,
    length: int = 80,
    n_back: int = 3,
    target_rate: float = 0.20,
    near_miss_rate: float = 0.10,
) -> dict:
    """
    Generate an n-back sequence with targets, near-miss distractors, and non-targets.

    - target: current letter == letter n positions back (correct response: YES)
    - near_miss: current letter is confusable with letter n positions back (correct: NO)
    - non_target: no match (correct: NO)

    Target rate decreases linearly from target_rate*1.2 to target_rate*0.8 across
    the sequence, simulating decreasing signal frequency (vigilance demand).
    """
    rng = random.Random(int(hashlib.sha256(seed.encode()).hexdigest(), 16))

    sequence = []
    letters = []

    for i in range(length):
        progress = i / length  # 0.0 → 1.0

        if i < n_back:
            # Initial n items: just random, no n-back possible
            letter = rng.choice(STIMULUS_POOL)
            item_type = "filler"
        else:
            ref_letter = letters[i - n_back]
            # Decrease target rate over time to increase vigilance demand
            local_target_rate = target_rate * (1.2 - 0.4 * progress)
            local_near_miss_rate = near_miss_rate * (0.8 + 0.4 * progress)  # increase near-misses over time

            roll = rng.random()
            if roll < local_target_rate:
                letter = ref_letter  # exact match = target
                item_type = "target"
            elif roll < local_target_rate + local_near_miss_rate:
                # near-miss: pick a confusable letter
                confusables = [c for c in CONFUSABLE.get(ref_letter, []) if c != ref_letter]
                if confusables:
                    letter = rng.choice(confusables)
                    item_type = "near_miss"
                else:
                    letter = rng.choice([s for s in STIMULUS_POOL if s != ref_letter])
                    item_type = "non_target"
            else:
                # non-target: any letter that isn't the ref or confusable
                avoid = set([ref_letter] + CONFUSABLE.get(ref_letter, []))
                pool = [s for s in STIMULUS_POOL if s not in avoid]
                if not pool:
                    pool = [s for s in STIMULUS_POOL if s != ref_letter]
                letter = rng.choice(pool)
                item_type = "non_target"

        letters.append(letter)

        quartile = i * 4 // length  # 0,1,2,3
        sequence.append({
            "position": i,
            "letter": letter,
            "type": item_type,
            "quartile": quartile,
            "n_back_ref": letters[i - n_back] if i >= n_back else None,
            "correct_response": "YES" if item_type == "target" else "NO",
        })

    stats = {
        "total": length,
        "n_back": n_back,
        "targets": sum(1 for s in sequence if s["type"] == "target"),
        "near_misses": sum(1 for s in sequence if s["type"] == "near_miss"),
        "non_targets": sum(1 for s in sequence if s["type"] == "non_target"),
        "fillers": sum(1 for s in sequence if s["type"] == "filler"),
    }

    return {
        "sequence": sequence,
        "n_back": n_back,
        "seed": seed,
        "stats": stats,
    }


# Pre-generate sequences for 3-back, 4-back, and 6-back conditions
VIGILANCE_3BACK = generate_nback_sequence("vig_3back_v2", length=80, n_back=3, near_miss_rate=0.15)
VIGILANCE_4BACK = generate_nback_sequence("vig_4back_v2", length=60, n_back=4, near_miss_rate=0.15)
VIGILANCE_6BACK = generate_nback_sequence("vig_6back_v2", length=80, n_back=6, near_miss_rate=0.18)

In [ ]:
"""
Attention Benchmark 2: Sustained Attention (Vigilance) — N-back Task

Tests sustained attention via n-back working memory monitoring over
long sequences with near-miss distractors.

Cognitive Science Basis:
- Kirchner (1958): N-back paradigm for working memory / sustained attention
- Mackworth (1948): Clock test — vigilance decrement over time
- Parasuraman & Davies (1977): Vigilance taxonomy and signal detection

Protocol:
1. Present sequence items one segment at a time (10 items per segment)
2. For each item at position i (where i >= n), model decides:
   "Is this letter the SAME as the letter n positions back?"
3. 3-back condition (80 items) + 4-back condition (60 items) + 6-back condition (80 items)
4. Near-miss distractors (confusable letters) increase false alarm rate
5. Target rate decreases over time → vigilance decrement

Scoring (composite):
  0.35 * overall_accuracy (hits + correct rejections)
  0.35 * sensitivity (hit_rate - false_alarm_rate, d' proxy)
  0.15 * vigilance_decrement_resistance (Q1 acc - Q4 acc, inverted)
  0.15 * (1 - false_alarm_rate)

Designed to break ceiling: near-miss distractors + long sequences +
decreasing target rate make perfect scores very unlikely.
"""

import kaggle_benchmarks as kbench
import re
import json



def _strip_think(text: str) -> str:
    """Remove <think>...</think> blocks from model output."""
    return re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()

def _parse_responses(raw: str, expected_count: int) -> list:
    """Extract YES/NO responses from model output."""
    raw = _strip_think(raw)
    raw = re.sub(r"//.*", "", raw)  # Strip JS-style comments
    # Try JSON array first
    try:
        m = re.search(r'\[.*\]', raw, re.DOTALL)
        if m:
            arr = json.loads(m.group())
            if len(arr) == expected_count:
                return [str(x).strip().upper() for x in arr]
    except Exception:
        pass

    # Try line-by-line or comma-separated
    tokens = re.findall(r'\b(YES|NO|yes|no|Yes|No|Y|N|y|n)\b', raw)
    result = []
    for t in tokens:
        t = t.upper()
        if t in ("Y", "YES"):
            result.append("YES")
        elif t in ("N", "NO"):
            result.append("NO")
    return result[:expected_count]


def run_nback_condition(llm, data: dict, condition_name: str) -> dict:
    """Run one n-back condition and return per-item results."""
    seq = data["sequence"]
    n = data["n_back"]
    segment_size = 10
    all_results = []

    # Build the full letter list for context
    letters = [item["letter"] for item in seq]

    for seg_start in range(0, len(seq), segment_size):
        seg_end = min(seg_start + segment_size, len(seq))
        seg_items = seq[seg_start:seg_end]

        # Only include items where n-back is possible
        eval_items = [item for item in seg_items if item["position"] >= n]
        if not eval_items:
            continue

        # Build the prompt showing the FULL sequence up to this segment
        # so the model has context for n-back lookups
        full_seq_so_far = letters[:seg_end]

        # Format: show positions with letters
        display_lines = []
        for i, letter in enumerate(full_seq_so_far):
            marker = " <-- respond" if seg_start <= i < seg_end and i >= n else ""
            display_lines.append(f"  [{i:2d}] {letter}{marker}")

        positions_to_judge = [item["position"] for item in eval_items]

        with kbench.chats.new(f"{condition_name}_seg{seg_start}"):
            prompt = (
                f"**{n}-Back Vigilance Task — Segment {seg_start // segment_size + 1}**\n\n"
                f"Rule: For each marked position, answer YES if the letter is the SAME as "
                f"the letter exactly {n} positions earlier. Answer NO otherwise.\n\n"
                f"Sequence so far:\n"
                + "\n".join(display_lines) + "\n\n"
                f"For positions {positions_to_judge}, respond with ONLY a JSON array of "
                f"YES/NO strings. Example: [\"YES\", \"NO\", \"NO\", ...]\n"
                f"Give exactly {len(eval_items)} responses."
            )

            raw = llm.prompt(prompt)
            responses = _parse_responses(raw, len(eval_items))

            # Pad if model gave too few
            while len(responses) < len(eval_items):
                responses.append("NO")  # default to NO (conservative)

            for item, resp in zip(eval_items, responses):
                hit = resp == item["correct_response"]
                all_results.append({
                    "position": item["position"],
                    "letter": item["letter"],
                    "type": item["type"],
                    "quartile": item["quartile"],
                    "correct_response": item["correct_response"],
                    "model_response": resp,
                    "correct": hit,
                    "is_false_alarm": (resp == "YES" and item["correct_response"] == "NO"),
                    "is_hit": (resp == "YES" and item["correct_response"] == "YES"),
                    "is_miss": (resp == "NO" and item["correct_response"] == "YES"),
                })

    return all_results


@kbench.task(name="Sustained Vigilance")
def attention_vigilance(llm) -> float:
    """N-Back Sustained Attention (Vigilance) Benchmark.

    Runs 3-back (80 items), 4-back (60 items), and 6-back (80 items) conditions.

    Score = 0.35 * overall_accuracy + 0.35 * sensitivity (hit_rate - FA_rate)
    """
    conditions = [
        ("3-back", VIGILANCE_3BACK, 0.30),  # weight
        ("4-back", VIGILANCE_4BACK, 0.35),
        ("6-back", VIGILANCE_6BACK, 0.35),
    ]

    condition_scores = []

    for cond_name, cond_data, weight in conditions:
        results = run_nback_condition(llm, cond_data, cond_name)

        if not results:
            condition_scores.append(0.0)
            continue

        # Overall accuracy
        overall_acc = sum(1 for r in results if r["correct"]) / len(results)

        # Quartile accuracies for vigilance decrement
        q_accs = {}
        for q in range(4):
            q_items = [r for r in results if r["quartile"] == q]
            if q_items:
                q_accs[q] = sum(1 for r in q_items if r["correct"]) / len(q_items)
            else:
                q_accs[q] = 0.0

        # Vigilance decrement = Q1 acc - Q4 acc (positive = decrement occurred)
        vig_decrement = max(0, q_accs.get(0, 0) - q_accs.get(3, 0))
        vig_resistance = 1.0 - vig_decrement

        # False alarm rate (said YES when should be NO)
        non_targets = [r for r in results if r["correct_response"] == "NO"]
        false_alarm_rate = (
            sum(1 for r in non_targets if r["is_false_alarm"]) / len(non_targets)
            if non_targets else 0.0
        )

        # d-prime proxy: hit_rate - false_alarm_rate (signal detection sensitivity)
        targets_list = [r for r in results if r["correct_response"] == "YES"]
        hit_rate_val = sum(1 for r in targets_list if r["is_hit"]) / len(targets_list) if targets_list else 0
        sensitivity = max(0, hit_rate_val - false_alarm_rate)

        cond_score = round(
            0.35 * overall_acc
            + 0.35 * sensitivity
            + 0.15 * vig_resistance
            + 0.15 * (1.0 - false_alarm_rate),
            4
        )

        condition_scores.append(cond_score * weight)

        # Logging
        n = cond_data["n_back"]
        targets = [r for r in results if r["correct_response"] == "YES"]
        hit_rate = sum(1 for r in targets if r["is_hit"]) / len(targets) if targets else 0
        miss_rate = sum(1 for r in targets if r["is_miss"]) / len(targets) if targets else 0

        print(f"\n{'='*60}")
        print(f"{cond_name.upper()} CONDITION RESULTS")
        print(f"{'='*60}")
        print(f"Items evaluated: {len(results)}")
        print(f"Overall accuracy: {overall_acc:.3f}")
        print(f"Hit rate (targets): {hit_rate:.3f}")
        print(f"Miss rate: {miss_rate:.3f}")
        print(f"False alarm rate: {false_alarm_rate:.3f}")
        print(f"Quartile accuracies: Q1={q_accs[0]:.3f} Q2={q_accs[1]:.3f} Q3={q_accs[2]:.3f} Q4={q_accs[3]:.3f}")
        print(f"Vigilance decrement: {vig_decrement:.3f}")
        print(f"Condition score: {cond_score:.4f} (weight={weight})")

    final_score = round(sum(condition_scores), 4)

    print(f"\n{'='*60}")
    print(f"FINAL VIGILANCE SCORE: {final_score:.4f}")
    print(f"{'='*60}")

    return min(1.0, max(0.0, final_score))

In [ ]:
attention_vigilance.run(llm=kbench.llm)